# Série Temporal de Consumo Elétrico — Aveiro (2024–2025)
**Input:**  — ficheiro já limpo, validado e com features geradas pelo notebook de pré-processamento.

Este notebook projeta o consumo horário para 2024–2025 com base nos dados reais de Janeiro e Fevereiro de 2024,
aplicando coeficientes sazonais mensais justificados, e apresenta a decomposição STL da série resultante.

In [1]:
# =============================================================
# CÉLULA 1 — IMPORTAÇÕES
# =============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


## 1. Carregar o Dataset Pré-processado

In [2]:
# Carregar directamente o ficheiro já limpo e validado
# Não é necessário carregar CSVs brutos, unir ficheiros nem limpar colunas —
# todo esse trabalho foi feito no notebook de pré-processamento.
df_aveiro = pd.read_csv("consumo_aveiro_preprocessado.csv", parse_dates=["data_hora"])
df_aveiro = df_aveiro.sort_values("data_hora").reset_index(drop=True)

# Colunas auxiliares necessárias para a projeção
df_aveiro["mes_real"]    = df_aveiro["data_hora"].dt.month
df_aveiro["dia_do_mes"]  = df_aveiro["data_hora"].dt.day
df_aveiro["hora_do_dia"] = df_aveiro["data_hora"].dt.hour

print(f"Registos carregados : {len(df_aveiro):,}")
print(f"Colunas disponíveis : {list(df_aveiro.columns)}")
print(f"Janela temporal     : {df_aveiro['data_hora'].min()} → {df_aveiro['data_hora'].max()}")


Registos carregados : 1,392
Colunas disponíveis : ['data_hora', 'ano', 'mes', 'dia', 'hora', 'dia_semana', 'dia_semana_nome', 'semana_ano', 'trimestre', 'fim_de_semana', 'estacao', 'periodo_dia', 'periodo_tarifario', 'energia_kwh', 'energia_minmax', 'energia_zscore', 'mes_real', 'dia_do_mes', 'hora_do_dia']
Janela temporal     : 2024-02-01 00:00:00+00:00 → 2024-02-29 23:00:00+00:00


## 2. Coeficientes Mensais de Sazonalidade

Fevereiro de 2024 é o mês âncora (coeficiente = 1.00) por ser o mês com mais dados reais disponíveis.
Todos os outros meses são escalados a partir do seu perfil horário.

| Mês | Coef. | Fundamento |
|-----|-------|------------|
| Jan | 1.25 | Inverno rigoroso; maior uso de aquecedores e bombas de calor |
| **Fev** | **1.00** | **Mês âncora — dados reais medidos na rede** |
| Mar | 1.05 | Fim do Inverno; redução gradual das cargas de aquecimento |
| Abr | 0.95 | Primavera temperada; consumo focado em eletrodomésticos base |
| Mai | 0.85 | Maior luz solar natural reduz iluminação artificial |
| Jun | 0.75 | Menor atividade escolar/universitária em Aveiro |
| Jul | 0.70 | Época balnear e férias reduzem consumo residencial regular |
| Ago | 0.65 | Mínimo anual: encerramento de empresas e menor densidade populacional |
| Set | 0.80 | Retorno da atividade industrial e arranque do ano letivo |
| Out | 0.95 | Outono; início de climatização pontual |
| Nov | 1.15 | Início do período crítico de frio; cargas de aquecimento sobem |
| Dez | 1.30 | Máximo anual: noites mais longas + frio extremo |

**Crescimento estrutural 2025:** fator adicional de +2%, alinhado com a taxa histórica de crescimento
do consumo elétrico em Portugal (ERSE, REN).


## 3. Projeção Horária 2024–2025

In [3]:
coeficientes = {
    1: 1.25, 2: 1.00, 3: 1.05, 4: 0.95, 5: 0.85, 6: 0.75,
    7: 0.70, 8: 0.65, 9: 0.80, 10: 0.95, 11: 1.15, 12: 1.30
}

# Calendário horário completo 2024–2025
horas_projetadas = pd.date_range(
    start="2024-01-01 00:00:00", end="2025-12-31 23:00:00", freq="h"
)
df_projeccao = pd.DataFrame({"data_hora": horas_projetadas})
df_projeccao["ano"]  = df_projeccao["data_hora"].dt.year
df_projeccao["mes"]  = df_projeccao["data_hora"].dt.month
df_projeccao["dia"]  = df_projeccao["data_hora"].dt.day
df_projeccao["hora"] = df_projeccao["data_hora"].dt.hour

# Pré-calcular médias horárias de Fevereiro como fallback eficiente
media_por_hora = (
    df_aveiro[df_aveiro["mes_real"] == 2]
    .groupby("hora_do_dia")["energia_kwh"]
    .mean()
    .to_dict()
)

valores_horarios = []

for _, row in df_projeccao.iterrows():
    dia_alvo  = int(row["dia"])
    mes_alvo  = int(row["mes"])
    ano_alvo  = int(row["ano"])
    hora_alvo = int(row["hora"])

    # Jan e Fev 2024: usar dados reais do ficheiro pré-processado
    if ano_alvo == 2024 and mes_alvo in [1, 2]:
        mask = (
            (df_aveiro["mes_real"]    == mes_alvo) &
            (df_aveiro["dia_do_mes"]  == dia_alvo) &
            (df_aveiro["hora_do_dia"] == hora_alvo)
        )
        vals = df_aveiro.loc[mask, "energia_kwh"].values
        if len(vals) > 0:
            valores_horarios.append(float(vals[0]))
            continue

    # Restantes meses: molde de Fevereiro + coeficiente sazonal
    # Dias 29/30/31 mapeiam para o último dia disponível de Fevereiro
    if dia_alvo > 28:
        dia_busca = 29 if (mes_alvo == 2 and dia_alvo == 29) else 28
    else:
        dia_busca = dia_alvo

    mask = (
        (df_aveiro["mes_real"]    == 2) &
        (df_aveiro["dia_do_mes"]  == dia_busca) &
        (df_aveiro["hora_do_dia"] == hora_alvo)
    )
    vals = df_aveiro.loc[mask, "energia_kwh"].values
    consumo_base = float(vals[0]) if len(vals) > 0 else media_por_hora.get(hora_alvo, 0.0)

    coef        = coeficientes[mes_alvo]
    fator_anual = 1.02 if ano_alvo == 2025 else 1.00

    valores_horarios.append(consumo_base * coef * fator_anual)

df_projeccao["energia_ativa_kwh"] = valores_horarios
print(f"Projeção concluída: {len(df_projeccao):,} registos horários (2024–2025).")


Projeção concluída: 17,544 registos horários (2024–2025).


## 4. Guardar a Série Horária

In [4]:
df_para_guardar = df_projeccao[["data_hora", "ano", "mes", "dia", "hora", "energia_ativa_kwh"]].copy()
df_para_guardar["chave_mes_ano"] = df_para_guardar["data_hora"].dt.strftime("%Y-%m")
df_para_guardar["dia_da_semana"] = df_para_guardar["data_hora"].dt.dayofweek  # 0=Segunda, 6=Domingo

df_para_guardar.to_csv("serie_projeccao_consumo_horario_2024_2025_aveiro.csv", index=False)

print("--- Ficheiro Horário Guardado com Sucesso! ---")
print(f"Total de horas gravadas : {len(df_para_guardar):,} linhas")
print(f"Colunas                 : {list(df_para_guardar.columns)}")


--- Ficheiro Horário Guardado com Sucesso! ---
Total de horas gravadas : 17,544 linhas
Colunas                 : ['data_hora', 'ano', 'mes', 'dia', 'hora', 'energia_ativa_kwh', 'chave_mes_ano', 'dia_da_semana']


## 5. Visualização e Decomposição Sazonal

In [5]:
# Agregação diária para os gráficos
df_diario_grafico = (
    df_projeccao.set_index("data_hora")["energia_ativa_kwh"]
    .resample("D").sum()
    .reset_index()
)
df_diario_grafico.columns = ["data", "Energia Ativa"]
df_diario_grafico["variacao_abs"] = df_diario_grafico["Energia Ativa"].diff()

df_diario_grafico.to_csv("serie_temporal_aveiro_limpa.csv", index=False)
print(f"-> Série unificada: {len(df_diario_grafico)} dias (Reais + Projetados).")

# --- Gráfico: Tendência e Desvios 2024–2025 ---
fig_trends = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=(
        "Evolução Contínua do Consumo em Aveiro (2024–2025)",
        "Desvio Diário Face ao Dia Anterior (DoD)"
    )
)
fig_trends.add_trace(
    go.Scatter(x=df_diario_grafico["data"], y=df_diario_grafico["Energia Ativa"],
               mode="lines", name="Consumo Diário",
               line=dict(color="#0284C7", width=1.5)),
    row=1, col=1
)
cores_barras = ["#0284C7" if v >= 0 else "#EA580C" for v in df_diario_grafico["variacao_abs"]]
fig_trends.add_trace(
    go.Bar(x=df_diario_grafico["data"], y=df_diario_grafico["variacao_abs"],
           marker_color=cores_barras, name="Desvio"),
    row=2, col=1
)
fig_trends.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                         height=500, showlegend=False)
fig_trends.update_xaxes(showgrid=False, linecolor="#E2E8F0")
fig_trends.update_yaxes(showgrid=True, gridcolor="#F1F5F9")
fig_trends.show()

# --- Decomposição Sazonal ---
df_decomp = df_diario_grafico.set_index("data")
analise_stl = seasonal_decompose(df_decomp["Energia Ativa"], model="additive", period=7)

fig_stl = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=(
        "1. Sinal Combinado Total (Dados Reais + Projeções)",
        "2. Tendência Macroscópica Isolada",
        "3. Padrão Sazonal Semanal (período = 7 dias)",
        "4. Resíduos / Ruído Irregular"
    )
)
fig_stl.add_trace(go.Scatter(x=analise_stl.observed.index, y=analise_stl.observed,
    mode="lines", line=dict(color="#475569", width=1.2)), row=1, col=1)
fig_stl.add_trace(go.Scatter(x=analise_stl.trend.index, y=analise_stl.trend,
    mode="lines", line=dict(color="#2563EB", width=2)), row=2, col=1)
fig_stl.add_trace(go.Scatter(x=analise_stl.seasonal.index, y=analise_stl.seasonal,
    mode="lines", line=dict(color="#0D9488", width=1.5)), row=3, col=1)
fig_stl.add_trace(go.Scatter(x=analise_stl.resid.index, y=analise_stl.resid,
    mode="markers", marker=dict(color="#EA580C", size=2)), row=4, col=1)
fig_stl.add_hline(y=0, line_dash="dot", line_color="#CBD5E1", row=4, col=1)

fig_stl.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                      height=650, showlegend=False, margin=dict(t=40, b=20))
for i in range(1, 5):
    fig_stl.update_xaxes(showgrid=False, linecolor="#E2E8F0", row=i, col=1)
    fig_stl.update_yaxes(showgrid=True,  gridcolor="#F1F5F9", row=i, col=1)
fig_stl.show()


-> Série unificada: 731 dias (Reais + Projetados).
